# Case study: steel EWS

The demo notebook. It produces **one dated, falsifiable example**:

1. Load the target series (`SLX`) and the Federal Register event table.
2. Produce a calibrated forecast interval (Chronos-2 + conformal).
3. Show an interval breach around a real tariff notice, with the attributed driver.
4. Report the **lead time** the system would have provided.

No results are committed until the pipeline has actually been run; the cells below
are the scaffold for that run.

In [ ]:
import yaml
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
cfg = yaml.safe_load(open(Path('..') / 'config' / 'steel.yaml'))
cfg['vertical']

In [ ]:
from src.data import prices as prices_mod
from src.data import events as events_mod

panel = prices_mod.build_price_panel(cfg)
events = events_mod.build_event_table(cfg)
panel.tail(), events.tail()

In [ ]:
from src.forecast.chronos_model import ChronosForecaster
from src.forecast.conformal import ConformalCalibrator, empirical_coverage

target = panel[cfg['target']['symbol']].dropna()
cal = ConformalCalibrator(ChronosForecaster(),
                          target_coverage=cfg['calibration']['target_coverage'],
                          calib_fraction=cfg['calibration']['calib_fraction'])
interval = cal.predict(target.iloc[:-cfg['forecast']['horizon_days']],
                       cfg['forecast']['horizon_days'], cfg['forecast']['quantiles'])
interval.to_frame()

In [ ]:
# Plot the interval + actuals, mark the breach, annotate the attributed event.
# Save to ../outputs/figures/ for the README worked example.
import matplotlib.pyplot as plt  # noqa: F401